# 10 · Phase 10 — The configuration MDP and its dynamic-programming solution (24AIM304 Unit 2)

**Finite MDP** (horizon H = 5, γ = 1):

| element | definition |
|---|---|
| state s_t | (context k, components chosen so far a_0 … a_{t−1}) — 396 states per context |
| action a_t | option of component t: wavelet (5) → level (3) → band (5) → window (4) → fusion (7) |
| transition | deterministic extension of the prefix; the *recording* drawn at s_0 is the randomness |
| reward | 0 for t < 4; r_i(config) (Notebooks 06–07) after the 5th decision |

Decomposing the 2,100-way choice into 5 small sequential choices is what gives the
tabular methods structure to exploit: the value of "db6" is shared across its 420 completions.

With the full empirical model (mean reward of every (context, configuration) pair — i.e. **all
2,100 × N_train reward queries**) the optimal policy follows from **value iteration** or
**policy iteration**. This is the planning optimum that the sample-based learners of
Notebooks 11–12 try to reach with far fewer reward queries.

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
import matplotlib.pyplot as plt

K = load_json(P["results"] / "chosen_K.json")["K"]
prob = RLProblem(CFG, P, K=K)
mdp = prob.mdp("train")
print(f"K={K} contexts; states per context={sum(mdp.P[:5])}; terminal configs={mdp.P[5]}; "
      f"exhaustive reward queries={mdp.P[5] * len(prob.y['train']):,}")

In [ ]:
t0 = time.time(); vi = value_iteration(mdp); t_vi = time.time() - t0
t0 = time.time(); pi = policy_iteration(mdp, np.random.default_rng(0)); t_pi = time.time() - t0
print(f"value iteration: {vi['sweeps']} sweeps ({t_vi * 1000:.1f} ms); max|ΔV| per sweep: {np.round(vi['deltas'], 5)}")
print(f"policy iteration: {pi['iterations']} improvement steps ({t_pi * 1000:.1f} ms)")
print("same optimal policy:", bool((vi["policy"] == pi["policy"]).all()))
prob.describe_policy(vi["policy"])

In [ ]:
rows = []
for nm, pol in [("baseline (fixed)", np.full(K, prob.space.baseline)), ("DP optimum (K contexts)", vi["policy"]),
                ("DP optimum (K=1, one global config)", np.full(K, value_iteration(RLProblem(CFG, P, K=1).mdp("train"))["policy"][0]))]:
    for s in ("train", "val"):
        m = prob.evaluate(pol, s)
        rows.append({"policy": nm, "split": s, **{k: m[k] for k in ("mean_reward", "macro_f1", "sensitivity", "specificity", "balanced_acc", "auroc")}})
dp_tab = pd.DataFrame(rows)
dp_tab.to_csv(P["tables"] / "10_dp_policies.csv", index=False)
dp_tab.round(4)

**Reading the table.** The DP-optimal configuration policy clearly beats the fixed baseline
configuration on held-out validation recordings (macro-F1 0.606 → 0.665, AUROC 0.727 → 0.794),
so the representation choice matters. The train→validation gap of the optimum (mean Δ-reward
0.057 → 0.052) is the **maximisation bias** of taking an arg-max over 2,100 noisy estimates; it
is small because the reward is out-of-fold. Adaptation across K = 8 contexts adds little over one
global configuration on validation (0.0520 vs 0.0519): most of the gain comes from choosing the
right *global* representation, the rest from context-specific choices.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
leaf = mdp.Rbar.reshape(K, *mdp.dims)
ax[0].imshow(leaf.max(axis=(2, 3, 4)).reshape(K, -1), aspect="auto", cmap="viridis")
ax[0].set_yticks(range(K)); ax[0].set_ylabel("context")
ax[0].set_xticks(range(15)); ax[0].set_xticklabels([f"{w}\nL{l}" for w in CFG["actions"]["wavelet"] for l in CFG["actions"]["level"]], fontsize=7)
ax[0].set_title("V*(k, wavelet, level): best achievable Δ-reward")
ax[1].plot(vi["deltas"], "o-"); ax[1].set_yscale("symlog", linthresh=1e-6); ax[1].set_xlabel("sweep"); ax[1].set_title("value-iteration convergence (exact after H sweeps)")
plt.tight_layout(); savefig(fig, "10_dp"); plt.show()
save_json({"policy": [int(c) for c in vi["policy"]], "K": K}, P["results"] / "policies" / "dp_optimum.json")